In [0]:


storage_account_name = "energysentineldl"
storage_account_key = "OiRgE+cLxjUGAE2GvfkNRuiIV04cs6kvxHzKI3nVcOU4xE4PpYrWImVDp7Szdh/JGJJUFXyBKCW5+AStLuP97Q=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

dbutils.fs.ls(f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/")
print(" Connected ")


✅ Connected!


In [0]:
# Databricks notebook source
import pickle
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)

# ---------------------------------------------------
# Config
STORAGE_ACCOUNT = "energysentineldl"    

EVENTHUB_NAMESPACE = "energysentinel"
EVENTHUB_NAME = "energy-sensors"
EVENTHUB_CONNECTION_STR = "Endpoint=sb://energysentinel.servicebus.windows.net/;SharedAccessKeyName=RootManageSharedAccessKey;SharedAccessKey=NE6EQCWRqydBHQXx/oPDrnjIPEtbD8Toi+AEhG/pKhU="


BRONZE_STREAM_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/streaming_raw"
BRONZE_CHECKPOINT = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/_checkpoints/streaming_raw"

PLATINUM_PATH = f"abfss://platinum@{STORAGE_ACCOUNT}.dfs.core.windows.net/ml_anomalies"
PLATINUM_CHECKPOINT = f"abfss://platinum@{STORAGE_ACCOUNT}.dfs.core.windows.net/_checkpoints/ml_anomalies"

MODEL_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/model/lgbm_model.pkl"
SCALER_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/model/scaler.pkl"

# ---------------------------------------------------
# الاتصال بـ Event Hub عن طريق بروتوكول Kafka

KAFKA_BOOTSTRAP_SERVERS = f"{EVENTHUB_NAMESPACE}.servicebus.windows.net:9093"

sasl_config = (
    f'org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" password="{EVENTHUB_CONNECTION_STR}";'
)

df_raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", EVENTHUB_NAME)
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.jaas.config", sasl_config)
    .option("startingOffsets", "earliest")
    .load()
)

# ---------------------------------------------------
# Parse الـ Body (JSON) من رسايل Kafka
reading_schema = StructType([
    StructField("sensor_id", StringType(), True),
    StructField("plant_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("power_consumption", DoubleType(), True),
    StructField("voltage", DoubleType(), True),
    StructField("current", DoubleType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("power_factor", DoubleType(), True),
    StructField("frequency", DoubleType(), True),
    StructField("is_anomaly", IntegerType(), True),
])

df_parsed = (
    df_raw_stream
    .withColumn("body_str", F.col("value").cast("string"))
    .withColumn("data", F.from_json(F.col("body_str"), reading_schema))
    .select("data.*")
    .withColumn("timestamp", F.to_timestamp("timestamp"))
    .withColumn("bronze_ingestion_ts", F.current_timestamp())
)

# ---------------------------------------------------
# 1) كتابة الداتا الخام في Bronze (كما هي)
bronze_query = (
    df_parsed.writeStream
    .format("delta")
    .option("checkpointLocation", BRONZE_CHECKPOINT)
    .outputMode("append")
    .start(BRONZE_STREAM_PATH)
)

# ---------------------------------------------------
# 2) تحميل الـ ML Model + Scaler

dbutils.fs.cp(
    "abfss://bronze@energysentineldl.dfs.core.windows.net/model/lgbm_model.pkl",
    "file:/tmp/lgbm_model.pkl"
)
dbutils.fs.cp(
    "abfss://bronze@energysentineldl.dfs.core.windows.net/model/scaler.pkl",
    "file:/tmp/scaler.pkl"
)

import joblib

model = joblib.load("/tmp/lgbm_model.pkl")
scaler = joblib.load("/tmp/scaler.pkl")

model_broadcast = sc.broadcast(model)
scaler_broadcast = sc.broadcast(scaler)

FEATURE_COLS = ["power_consumption", "voltage", "current", "temperature", "power_factor", "frequency"]

# ---------------------------------------------------
# 3) دالة تشغيل الـ ML على كل Micro-Batch
def predict_anomaly(pdf: pd.DataFrame) -> pd.DataFrame:
    if pdf.empty:
        pdf["ml_predicted_anomaly"] = []
        pdf["ml_anomaly_probability"] = []
        return pdf

    X = pdf[FEATURE_COLS]
    X_scaled = scaler_broadcast.value.transform(X)

    predictions = model_broadcast.value.predict(X_scaled)
    probabilities = model_broadcast.value.predict_proba(X_scaled)[:, 1]

    pdf["ml_predicted_anomaly"] = predictions
    pdf["ml_anomaly_probability"] = probabilities
    return pdf


def process_batch_for_ml(batch_df, batch_id):
    if batch_df.rdd.isEmpty():
        return

    pdf = batch_df.toPandas()
    result_pdf = predict_anomaly(pdf)

    result_df = spark.createDataFrame(result_pdf)
    (
        result_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .save(PLATINUM_PATH)
    )

# ---------------------------------------------------
# 4) تشغيل الـ ML Stream على نفس الداتا

platinum_query = (
    df_parsed.writeStream
    .foreachBatch(process_batch_for_ml)
    .option("checkpointLocation", PLATINUM_CHECKPOINT)
    .outputMode("append")
    .start()
)

# ---------------------------------------------------
# استنى الـ Streams يفضلوا شغالين
bronze_query.awaitTermination()

/databricks/python/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/d

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
raw_bytes = dbutils.fs.head("abfss://bronze@energysentineldl.dfs.core.windows.net/model/lgbm_model.pkl", 20)
print(raw_bytes)

[Truncated to first 20 bytes]
���      �lightgb


In [0]:
import pandas as pd

test_row = pd.DataFrame([{
    "power_consumption": 500.0,
    "voltage": 220.0,
    "current": 2.3,
    "temperature": 70.0,
    "power_factor": 0.87,
    "frequency": 50.0,
}])

test_scaled = scaler.transform(test_row)
prediction = model.predict(test_scaled)
probability = model.predict_proba(test_scaled)

print("Prediction:", prediction)
print("Probability:", probability)

Prediction: [0]
Probability: [[9.99997727e-01 2.27271196e-06]]


/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [0]:
display(spark.read.format("delta").load(PLATINUM_PATH).orderBy(F.desc("bronze_ingestion_ts")).limit(10))

sensor_id,plant_id,timestamp,power_consumption,voltage,current,temperature,power_factor,frequency,is_anomaly,bronze_ingestion_ts,ml_predicted_anomaly,ml_anomaly_probability
SENSOR_A01,PLANT_A,2026-07-07T02:36:24.616246Z,516.67,218.68,2.48,73.45,0.887,50.07,0,2026-07-07T02:36:25.676Z,0,6.1543787696374E-5
SENSOR_A04,PLANT_A,2026-07-07T02:36:24.616246Z,491.96,219.37,1.82,69.4,0.864,50.04,0,2026-07-07T02:36:25.676Z,0,2.272711959423284E-6
SENSOR_A03,PLANT_A,2026-07-07T02:36:24.616246Z,542.28,218.49,2.5,74.85,0.863,50.05,0,2026-07-07T02:36:25.676Z,0,6.1543787696374E-5
SENSOR_B03,PLANT_B,2026-07-07T02:36:24.616246Z,290.49,220.13,0.95,66.31,0.884,49.98,0,2026-07-07T02:36:25.676Z,0,6.710065834394354E-4
SENSOR_A05,PLANT_A,2026-07-07T02:36:24.616246Z,539.01,219.38,2.66,75.32,0.882,50.08,0,2026-07-07T02:36:25.676Z,1,0.9999468730768248
SENSOR_C01,PLANT_C,2026-07-07T02:36:24.616246Z,120.48,218.68,0.24,51.43,0.889,49.91,0,2026-07-07T02:36:25.676Z,0,3.4133617252030236E-5
SENSOR_B02,PLANT_B,2026-07-07T02:36:24.616246Z,243.12,219.02,1.29,62.03,0.851,49.94,0,2026-07-07T02:36:25.676Z,0,0.4905935709868146
SENSOR_B01,PLANT_B,2026-07-07T02:36:24.616246Z,262.19,220.96,1.02,64.65,0.859,49.98,0,2026-07-07T02:36:25.676Z,0,0.001170554404127761
SENSOR_A02,PLANT_A,2026-07-07T02:36:24.616246Z,499.28,220.39,2.67,70.67,0.852,50.08,0,2026-07-07T02:36:25.676Z,0,8.234872340880025E-6
SENSOR_C02,PLANT_C,2026-07-07T02:36:24.616246Z,107.08,218.6,0.9,50.75,0.867,50.09,0,2026-07-07T02:36:25.676Z,0,2.125910370161647E-6
